# Haiku 4.5 -- Combined Batch: ZS-CoT + FS-CoT + Self-Consistency

700 requests in a single **Anthropic Message Batch**:
- **100** Zero-Shot CoT (v2 format-hinted prompt) -- temp=0
- **100** Few-Shot CoT (8 canonical exemplars) -- temp=0
- **500** Self-Consistency (5 paths x 100 questions) -- temp=0.7

**Model:** `claude-haiku-4-5-20251001`
**Seed:** 42 -> identical 100 GSM8K problems to the Llama 3.1 8B runs
**Pricing:** Haiku 4.5 = $1/M input, $5/M output; batch discount 50% = $0.50/M input, $2.50/M output
**Estimated cost:** ~$0.50-1.00 (700 batch requests, short prompts)
**Estimated time:** typically 5-30 min (max 24h SLA)

## Prerequisites
1. **Secrets:** add `ANTHROPIC_API_KEY`
2. **Drive:** upload `MyDrive/NLP_Haiku/data/gsm8k_test.json` (the same file used by the Llama runs)
3. Few-shot exemplars are inlined into the notebook -- no extra upload required

## Output files
- `MyDrive/NLP_Haiku/results/zero_shot_cot_haiku.json`
- `MyDrive/NLP_Haiku/results/few_shot_cot_haiku.json`
- `MyDrive/NLP_Haiku/results/self_consistency_haiku.json`
- `MyDrive/NLP_Haiku/results/batch_id.txt` (used to resume polling if Colab disconnects)


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE    = '/content/drive/MyDrive/NLP_Haiku'
DRIVE_DATA    = os.path.join(DRIVE_BASE, 'data')
DRIVE_RESULTS = os.path.join(DRIVE_BASE, 'results')

os.makedirs(DRIVE_DATA,    exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Drive mounted.')
print('Data dir   :', DRIVE_DATA)
print('Results dir:', DRIVE_RESULTS)

In [ ]:
# 2. Install the Anthropic SDK
!pip install -q anthropic
import anthropic
print('anthropic', anthropic.__version__)

In [ ]:
# 3. Load API key from Colab Secrets
from google.colab import userdata
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY').strip()
if not ANTHROPIC_API_KEY:
    raise ValueError('ANTHROPIC_API_KEY not found. Add it from the Secrets panel.')
print('API key loaded.')

In [ ]:
# 4. Configuration
MODEL       = 'claude-haiku-4-5-20251001'
N_SAMPLES   = 100
SC_PATHS    = 5
SEED        = 42
MAX_TOKENS  = 1024
TEMP_ZS     = 0.0   # Zero-Shot CoT (greedy)
TEMP_FS     = 0.0   # Few-Shot CoT (greedy)
TEMP_SC     = 0.7   # Self-Consistency (diverse sampling)

DATA_FILE        = os.path.join(DRIVE_DATA,    'gsm8k_test.json')
ZS_RESULT_FILE   = os.path.join(DRIVE_RESULTS, 'zero_shot_cot_haiku.json')
FS_RESULT_FILE   = os.path.join(DRIVE_RESULTS, 'few_shot_cot_haiku.json')
SC_RESULT_FILE   = os.path.join(DRIVE_RESULTS, 'self_consistency_haiku.json')
BATCH_ID_FILE    = os.path.join(DRIVE_RESULTS, 'batch_id.txt')
REQUESTS_DUMP    = os.path.join(DRIVE_RESULTS, 'batch_requests_dump.json')

print(f'Model         : {MODEL}')
print(f'N samples     : {N_SAMPLES}')
print(f'SC paths      : {SC_PATHS}')
print(f'Total requests: {N_SAMPLES * (2 + SC_PATHS)}')

In [ ]:
# 5. Load GSM8K test data (seed=42 -> identical 100 problems to the Llama runs)
import json
import random

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Data file not found: {DATA_FILE}\n'
        'Please upload gsm8k_test.json to Drive/NLP_Haiku/data/.'
    )

with open(DATA_FILE, encoding='utf-8') as f:
    all_data = json.load(f)

random.seed(SEED)
test_data = random.sample(all_data, N_SAMPLES)

print(f'Total test set: {len(all_data)}')
print(f'This run      : {len(test_data)} problems (seed={SEED})')
print('\nFirst 120 characters of problem #0:')
print(test_data[0]['question'][:120], '...')

In [ ]:
# 6. Few-shot exemplars (identical 8 problems used by the Llama runs)
FEW_SHOT_EXAMPLES = [
    {
        "question": "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?",
        "answer": "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6 trees planted. The answer is 6."
    },
    {
        "question": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?",
        "answer": "There are originally 3 cars. Then 2 more cars arrive. So there are 3 + 2 = 5 cars now. The answer is 5."
    },
    {
        "question": "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        "answer": "Originally, Leah had 32 chocolates and her sister had 42. So in total they had 32 + 42 = 74 chocolates. After eating 35, they had 74 - 35 = 39 chocolates. The answer is 39."
    },
    {
        "question": "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?",
        "answer": "Jason started with 20 lollipops. He then gave some to Denny and ended up with 12. So he gave 20 - 12 = 8 lollipops to Denny. The answer is 8."
    },
    {
        "question": "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?",
        "answer": "Shawn started with 5 toys. He got 2 toys from his mom and 2 toys from his dad. So he got 2 + 2 = 4 more toys. In total he now has 5 + 4 = 9 toys. The answer is 9."
    },
    {
        "question": "There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?",
        "answer": "There were originally 9 computers. From Monday to Thursday is 4 days. So 4 * 5 = 20 new computers were added. In total, there are now 9 + 20 = 29 computers. The answer is 29."
    },
    {
        "question": "Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?",
        "answer": "Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33. The answer is 33."
    },
    {
        "question": "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?",
        "answer": "Olivia started with $23. She bought 5 bagels for $3 each, so she spent 5 * 3 = $15 on bagels. She now has 23 - 15 = $8 left. The answer is 8."
    },
]

FEW_SHOT_BLOCK = "\n\n".join(f"Q: {e['question']}\nA: {e['answer']}" for e in FEW_SHOT_EXAMPLES)
print(f'{len(FEW_SHOT_EXAMPLES)} few-shot exemplars loaded.')

In [ ]:
# 7. Prompt templates (v2 format hint shared by ZS and FS) + answer parser
import re

# ZS-CoT v2 prompt (identical to the Llama 8B v2 run)
ZS_PROMPT = (
    "Q: {question}\n"
    "A: Let's think step by step. "
    "At the end, write your final numeric answer in the format: "
    "\"The answer is X.\" where X is just a single number with no units."
)

def build_zs_prompt(q):
    return ZS_PROMPT.format(question=q)

def build_fs_prompt(q):
    return f"{FEW_SHOT_BLOCK}\n\nQ: {q}\nA:"

def extract_answer(text):
    """Same parser as the Llama v2 runs: #### marker > 'answer is X' > last number."""
    if not text:
        return None
    m = re.search(r'####\s*\$?([\d,]+\.?\d*)', text)
    if m:
        try: return float(m.group(1).replace(',', ''))
        except: pass
    matches = re.findall(
        r'(?:final\s+)?answer\s+is\s*[:\-]?\s*\$?([\d,]+\.?\d*)',
        text, re.IGNORECASE
    )
    if matches:
        try: return float(matches[-1].replace(',', ''))
        except: pass
    nums = re.findall(r'[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[-1]) if nums else None

def gold_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    return float(m.group(1).replace(',', '')) if m else None

print('Prompts and parser ready.')
print('\nSample ZS prompt:')
print(build_zs_prompt(test_data[0]['question'])[:240], '...')

In [ ]:
# 8. Build the 700 batch requests
# custom_id layout:
#   zs_q{i}       -> Zero-Shot CoT, question i
#   fs_q{i}       -> Few-Shot CoT, question i
#   sc_q{i}_p{j}  -> Self-Consistency, question i, path j (j in 0..SC_PATHS-1)
requests = []

for i, item in enumerate(test_data):
    q = item['question']

    # Zero-Shot CoT
    requests.append({
        'custom_id': f'zs_q{i}',
        'params': {
            'model':       MODEL,
            'max_tokens':  MAX_TOKENS,
            'temperature': TEMP_ZS,
            'messages':    [{'role': 'user', 'content': build_zs_prompt(q)}],
        },
    })

    # Few-Shot CoT
    requests.append({
        'custom_id': f'fs_q{i}',
        'params': {
            'model':       MODEL,
            'max_tokens':  MAX_TOKENS,
            'temperature': TEMP_FS,
            'messages':    [{'role': 'user', 'content': build_fs_prompt(q)}],
        },
    })

    # Self-Consistency (5 paths, same FS prompt, temp=0.7)
    for j in range(SC_PATHS):
        requests.append({
            'custom_id': f'sc_q{i}_p{j}',
            'params': {
                'model':       MODEL,
                'max_tokens':  MAX_TOKENS,
                'temperature': TEMP_SC,
                'messages':    [{'role': 'user', 'content': build_fs_prompt(q)}],
            },
        })

print(f'Total requests: {len(requests)}')
print(f'  ZS  : {sum(1 for r in requests if r["custom_id"].startswith("zs_"))}')
print(f'  FS  : {sum(1 for r in requests if r["custom_id"].startswith("fs_"))}')
print(f'  SC  : {sum(1 for r in requests if r["custom_id"].startswith("sc_"))}')

# Persist the full request list for audit
with open(REQUESTS_DUMP, 'w', encoding='utf-8') as f:
    json.dump({
        'model':     MODEL,
        'seed':      SEED,
        'n_samples': N_SAMPLES,
        'sc_paths':  SC_PATHS,
        'requests':  requests,
    }, f, indent=2, ensure_ascii=False)
print(f'Requests dump -> {REQUESTS_DUMP}')

In [ ]:
# 9. Submit the batch (skipped if a batch_id already exists)
from anthropic import Anthropic
client = Anthropic(api_key=ANTHROPIC_API_KEY, timeout=120.0)

if os.path.exists(BATCH_ID_FILE):
    with open(BATCH_ID_FILE) as f:
        BATCH_ID = f.read().strip()
    print(f'Existing batch_id found: {BATCH_ID}')
    print('(To submit a new batch, delete batch_id.txt and re-run this cell.)')
else:
    print(f'Submitting new batch... ({len(requests)} requests)')
    batch = client.messages.batches.create(requests=requests)
    BATCH_ID = batch.id
    with open(BATCH_ID_FILE, 'w') as f:
        f.write(BATCH_ID)
    print(f'Batch created     : {BATCH_ID}')
    print(f'Status            : {batch.processing_status}')
    print(f'Batch ID saved to : {BATCH_ID_FILE}')

In [ ]:
# 10. Poll until the batch finishes (status check every 30 seconds)
import time

POLL_INTERVAL = 30   # seconds
MAX_WAIT      = 6 * 3600   # 6 hours

elapsed = 0
while elapsed < MAX_WAIT:
    b = client.messages.batches.retrieve(BATCH_ID)
    status = b.processing_status
    counts = b.request_counts
    total  = counts.succeeded + counts.errored + counts.canceled + counts.expired + counts.processing
    print(f'  [{elapsed:>5}s] status={status:>11}  '
          f'succeeded={counts.succeeded:>4}  errored={counts.errored:>3}  '
          f'processing={counts.processing:>4}  total={total}', flush=True)
    if status == 'ended':
        print('\nBatch finished.')
        break
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    raise RuntimeError(f'Batch did not finish within {MAX_WAIT}s.')

In [ ]:
# 11. Download results, parse, and write three separate JSON files
from collections import Counter

# custom_id -> {text, input_tokens, output_tokens}
responses = {}
errors    = {}
print('Downloading results...')
for r in client.messages.batches.results(BATCH_ID):
    cid = r.custom_id
    if r.result.type == 'succeeded':
        msg = r.result.message
        text = next((b.text for b in msg.content if getattr(b, 'type', None) == 'text'), '') if msg.content else ''
        responses[cid] = {
            'text':         text,
            'input_tokens': msg.usage.input_tokens,
            'output_tokens': msg.usage.output_tokens,
        }
    else:
        err = getattr(r.result, 'error', None) or r.result.type
        errors[cid] = str(err)

print(f'  Succeeded: {len(responses)}')
print(f'  Errored  : {len(errors)}')
if errors:
    print('  First 5 errors:')
    for k in list(errors.keys())[:5]:
        print(f'    {k}: {errors[k][:150]}')

# ---------- Zero-Shot CoT ----------
zs_results = []
for i, item in enumerate(test_data):
    cid = f'zs_q{i}'
    resp = responses.get(cid, {}).get('text', '')
    predicted = extract_answer(resp)
    gold      = gold_answer(item['answer'])
    zs_results.append({
        'question':  item['question'],
        'gold':      gold,
        'predicted': predicted,
        'correct':   predicted == gold,
        'response':  resp,
        'usage':     {
            'input_tokens':  responses.get(cid, {}).get('input_tokens'),
            'output_tokens': responses.get(cid, {}).get('output_tokens'),
        },
        'error':     errors.get(cid),
    })

zs_correct = sum(r['correct'] for r in zs_results)
with open(ZS_RESULT_FILE, 'w', encoding='utf-8') as f:
    json.dump({
        'strategy':      'zero_shot_cot_v2_haiku',
        'model':         MODEL,
        'temperature':   TEMP_ZS,
        'n_done':        len(zs_results),
        'accuracy':      round(zs_correct / len(zs_results) * 100, 1),
        'prompt_format': 'v2 (let_s think step by step + answer is X hint)',
        'results':       zs_results,
    }, f, indent=2, ensure_ascii=False)
print(f'\nZS-CoT  : {zs_correct}/{len(zs_results)} = {zs_correct/len(zs_results)*100:.1f}%  -> {ZS_RESULT_FILE}')

# ---------- Few-Shot CoT ----------
fs_results = []
for i, item in enumerate(test_data):
    cid = f'fs_q{i}'
    resp = responses.get(cid, {}).get('text', '')
    predicted = extract_answer(resp)
    gold      = gold_answer(item['answer'])
    fs_results.append({
        'question':  item['question'],
        'gold':      gold,
        'predicted': predicted,
        'correct':   predicted == gold,
        'response':  resp,
        'usage':     {
            'input_tokens':  responses.get(cid, {}).get('input_tokens'),
            'output_tokens': responses.get(cid, {}).get('output_tokens'),
        },
        'error':     errors.get(cid),
    })

fs_correct = sum(r['correct'] for r in fs_results)
with open(FS_RESULT_FILE, 'w', encoding='utf-8') as f:
    json.dump({
        'strategy':    'few_shot_cot_haiku',
        'model':       MODEL,
        'temperature': TEMP_FS,
        'n_done':      len(fs_results),
        'accuracy':    round(fs_correct / len(fs_results) * 100, 1),
        'n_few_shot':  len(FEW_SHOT_EXAMPLES),
        'results':     fs_results,
    }, f, indent=2, ensure_ascii=False)
print(f'FS-CoT  : {fs_correct}/{len(fs_results)} = {fs_correct/len(fs_results)*100:.1f}%  -> {FS_RESULT_FILE}')

# ---------- Self-Consistency ----------
sc_results = []
for i, item in enumerate(test_data):
    sampled_responses = []
    sampled_answers   = []
    in_toks = out_toks = 0
    for j in range(SC_PATHS):
        cid = f'sc_q{i}_p{j}'
        resp = responses.get(cid, {}).get('text', '')
        sampled_responses.append(resp)
        sampled_answers.append(extract_answer(resp))
        in_toks  += responses.get(cid, {}).get('input_tokens')  or 0
        out_toks += responses.get(cid, {}).get('output_tokens') or 0

    valid  = [a for a in sampled_answers if a is not None]
    predicted = Counter(valid).most_common(1)[0][0] if valid else None
    gold = gold_answer(item['answer'])

    sc_results.append({
        'question':        item['question'],
        'gold':            gold,
        'predicted':       predicted,
        'correct':         predicted == gold,
        'sampled_answers': sampled_answers,
        'vote_counts':     dict(Counter(a for a in sampled_answers if a is not None)),
        'responses':       sampled_responses,
        'usage':           {'input_tokens': in_toks, 'output_tokens': out_toks},
    })

sc_correct = sum(r['correct'] for r in sc_results)
with open(SC_RESULT_FILE, 'w', encoding='utf-8') as f:
    json.dump({
        'strategy':    f'self_consistency_{SC_PATHS}paths_haiku',
        'model':       MODEL,
        'temperature': TEMP_SC,
        'n_runs':      SC_PATHS,
        'n_done':      len(sc_results),
        'accuracy':    round(sc_correct / len(sc_results) * 100, 1),
        'results':     sc_results,
    }, f, indent=2, ensure_ascii=False)
print(f'SC      : {sc_correct}/{len(sc_results)} = {sc_correct/len(sc_results)*100:.1f}%  -> {SC_RESULT_FILE}')

In [ ]:
# 12. Summary metrics: gold-correction + Cobbe stratified accuracy

def n_ops(gold_text):
    """Cobbe stratification proxy: count '=' in the gold answer (one per op)."""
    pre = gold_text.split('####')[0]
    return pre.count('=')

def stratified(results, items):
    buckets = {'easy': [], 'medium': [], 'hard': []}
    for r, item in zip(results, items):
        ops = n_ops(item['answer'])
        if ops < 3:    buckets['easy'].append(r['correct'])
        elif ops == 3: buckets['medium'].append(r['correct'])
        else:          buckets['hard'].append(r['correct'])
    return {k: (sum(v), len(v), round(sum(v)/len(v)*100, 1) if v else 0.0)
            for k, v in buckets.items()}

def gold_corrected_count(results):
    """Carnival annotation-error correction: gold says 2280, true answer is 2180."""
    fixed = sum(r['correct'] for r in results)
    for r in results:
        if 'carnival' in r.get('question', '').lower() and not r.get('correct'):
            if r.get('predicted') == 2180.0:
                fixed += 1
    return fixed

print('=== Results (raw + gold-corrected) ===\n')
for name, res, path in [
    ('ZS-CoT v2', zs_results, ZS_RESULT_FILE),
    ('FS-CoT',    fs_results, FS_RESULT_FILE),
    ('SC (5)',    sc_results, SC_RESULT_FILE),
]:
    n        = len(res)
    raw      = sum(r['correct'] for r in res)
    gc       = gold_corrected_count(res)
    strat    = stratified(res, test_data)
    print(f'{name}:')
    print(f'  Raw            : {raw}/{n} = {raw/n*100:.1f}%')
    print(f'  Gold-corrected : {gc}/{n} = {gc/n*100:.1f}%')
    print(f'  Easy           : {strat["easy"][2]}% ({strat["easy"][0]}/{strat["easy"][1]})')
    print(f'  Medium         : {strat["medium"][2]}% ({strat["medium"][0]}/{strat["medium"][1]})')
    print(f'  Hard           : {strat["hard"][2]}% ({strat["hard"][0]}/{strat["hard"][1]})')
    print()

# Llama 3.1 8B reference (for side-by-side comparison)
print('=== Llama 3.1 8B reference (side-by-side comparison) ===')
print('  ZS-CoT v2 : gold-corrected 95.0%  | Hard 94.6%')
print('  FS-CoT    : raw            85.0%  | (Llama)')
print('  SC (5)    : gold-corrected 92.0%  | Hard 86.5%')

In [ ]:
# 13. Token usage and realized cost
def sum_tokens(results, key):
    return sum((r.get('usage', {}) or {}).get(key) or 0 for r in results)

total_in  = sum_tokens(zs_results, 'input_tokens')  + sum_tokens(fs_results, 'input_tokens')  + sum_tokens(sc_results, 'input_tokens')
total_out = sum_tokens(zs_results, 'output_tokens') + sum_tokens(fs_results, 'output_tokens') + sum_tokens(sc_results, 'output_tokens')

print('Token usage summary:')
print(f'  Input  : {total_in:>10,} tok')
print(f'  Output : {total_out:>10,} tok')
print(f'  TOTAL  : {total_in + total_out:>10,} tok')

# Haiku 4.5 batch pricing: $0.50/M input, $2.50/M output (full price * 0.5)
COST = (total_in / 1e6) * 0.50 + (total_out / 1e6) * 2.50
print(f'\nEstimated cost (BATCH = full * 0.5): ${COST:.4f}')
print(f'  ({total_in:,} in @ $0.50/M  +  {total_out:,} out @ $2.50/M)')

# For reference: what would sync (full price) have cost?
SYNC_COST = (total_in / 1e6) * 1.00 + (total_out / 1e6) * 5.00
print(f'Reference (sync would cost): ${SYNC_COST:.4f}  -> batch savings ${SYNC_COST - COST:.4f}')